# Assignment 8 — Robustness Analysis

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  


---

## Learning Outcomes

After completing this assignment you will be able to:

1. Re-evaluate a set of Pareto-optimal policies across an ensemble of climate
   scenarios.

2. Compute and interpret **satisficing scores** — the fraction of scenarios in
   which a policy meets an acceptable threshold on all objectives simultaneously.

3. Compute and interpret **minimax regret** — the worst-case loss a policy
   incurs relative to the best achievable outcome in each scenario.

---

## Assignment Overview

This assignment re-evaluates your Pareto-optimal policies by re-running JUSTICE across multiple climate scenarios (FAIR ensemble members) and applying two complementary robustness metrics:

- **Satisficing** — does the policy meet an acceptable threshold on every objective, across enough scenarios?
- **Minimax regret** — which policy has the smallest worst-case loss relative to what was achievable?

**What you will do**

- Run `run_reeval.py` to evaluate every policy under multiple climate scenarios
- Apply satisficing analysis — choose thresholds, compute scores, and visualise
- Apply minimax regret — compute worst-case performance 

**What you will need**

- Your **reference set** from Assignment 6 (the Pareto-optimal solutions)
- Your **config file** the one you used to run the optimization or optimizations

**What you will produce**

- A heatmap showing which objectives are hardest to satisfy
- A maximum regret ranking of all policies
- A CDF plot of worst-case regret across the Pareto front
- Your recommended solution based on the analysis

> **Scope note:** This is a simplified robustness analysis. A full robustness
> analysis would test policy performance across uncertainty in many parameters
> simultaneously, you can think of all the model assumptions, economic inputs, damage functions, and
> more. Here we vary only the climate uncertainty (FAIR ensemble members),
> holding everything else fixed. This is a meaningful first step, but keep in
> mind that the robustness scores you compute reflect climate uncertainty only,
> not the full uncertainty space the model operates in.

___

## Setup — Imports, paths, and model constants

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as _mpath
import seaborn as sns

# ── Patch matplotlib Path deepcopy ────────────────────────────────────────────
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib; matplotlib.use("Agg")

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

# ── Paths ─────────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))
_PLOTS_DIR   = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)

# ── Objectives ────────────────────────────────────────────────────────────────
OBJECTIVES = ["welfare", "years_above_2C", "welfare_loss_damage", "welfare_loss_abatement"]
OBJ_LABELS = ["Welfare", "Yrs > 2°C", "WL Damage", "WL Abatement"]


print(f"Results root : {RESULTS_ROOT}")
print("Setup OK")


Results root : c:\Users\matsm\epa141a\assignments_ema\results
Setup OK


## Step 1 — Re-evaluation using the EMA Workbench

To assess the robustness of your Pareto-optimal policies, you need to re-run JUSTICE for every policy under multiple climate scenarios. This is a computationally expensive operation — running it sequentially inside a notebook would take hours on a single core.

Instead, you can adapt the provided script [assignments_ema/run_reeval.py](assignments_ema/run_reeval.py), which runs the experiment in parallel using all available CPU cores and saves the results.

### 1.1. What the script does

The script uses the EMA Workbench (`perform_experiments`) to handle the full factorial experiment — every policy × every scenario.

These are the steps implemented in the script:

1. **Set up paths** — figures out where it lives on disk and builds paths to the JUSTICE model, the config file, and the results folder.

2. **Load your config file** — reads your model settings (start year, end year, timestep, scenario index, etc.) — the same settings used during optimisation.

3. **Compute model constants** — determines how many timesteps and regions the model has, and what shape the RBF parameters take (8 centers, 8 radii, 228 weights).

4. **Define the model wrapper** — wraps JUSTICE in a plain Python function that EMA Workbench calls once per (policy, scenario) pair. It must be at the top level of the script — not inside a notebook cell — so Python's multiprocessing system can distribute it to worker processes. Each call:
   - Receives the 244 RBF parameters and the climate ensemble index as inputs
   - Rebuilds the RBF and runs JUSTICE forward through time (identical to Assignment 7)
   - Returns the four objectives: welfare, years above 2°C, welfare loss from damage, and welfare loss from abatement

5. **Load the reference set** — reads your reference set CSV, and identifies the 244 lever columns (this is essentially what you need for the reruns).

6. **Check if results already exist** — if the output files are already on disk, the script exits immediately so you don't accidentally re-run a long computation.

7. **Register the model with EMA Workbench** — creates a `Model` object and declares:
   - **Uncertainties** — `climate_ensemble_index` (integer 1–1000): which FAIR climate trajectory to use. Varies across scenarios.
   - **Levers** — the 244 RBF parameters (centers, radii, weights). Vary across policies, fixed within each policy.
   - **Outcomes** — the four scalar objectives.

> **Note on objective directions:** In Assignment 4, `welfare_loss_damage` and
> `welfare_loss_abatement` were declared as `MAXIMIZE` during optimisation — this
> was an EMA Workbench convention for handling positive quantities, not a sign
> that higher losses are better. Here, with no optimiser involved, they are
> correctly declared as `MINIMIZE`: lower welfare loss is always better.

8. **Build policy and scenario lists** — creates one `Policy` object per row in the reference set and one `Scenario` object per selected FAIR ensemble member.

9. **Run in parallel** — distributes all (policy, scenario) combinations across CPU cores using `MultiprocessingEvaluator`.

10. **Reshape and save** — reorganises the results into two files:
    - **`<run_name>_<n_policies>p_<n_scenarios>s.npy`** — a 3D NumPy array of shape `(n_policies, n_scenarios, n_objectives)` with the four objective values for every combination. This is what the notebook uses for robustness analysis.
    - **`<run_name>_<n_policies>p_<n_scenarios>s_experiments.csv`** — a flat table recording the inputs (policy, scenario, lever values) for every run.


### 1.2. Running the script

**Step 1 — Update the script**

Open [assignments_ema/run_reeval.py](assignments_ema/run_reeval.py) and make the necessary changes, at the very least, you'll need to:

1. Replace `config_ssp245.json` with your own config
2. Replace `reference_set_utilitarian.csv` with your reference set.

**Step 2 — Run from the terminal**

Open a terminal in the `epa141a` folder and run:

```bash
# Quick test — verify everything works before a full run
.venv/bin/python assignments_ema/run_reeval.py --n_scenarios 10

# Full run (this will take several hours)
.venv/bin/python assignments_ema/run_reeval.py --n_scenarios 1000

# Limit cores if needed (e.g. leave one free for other work)
.venv/bin/python assignments_ema/run_reeval.py --n_scenarios 1000 --n_cores 4
```

> Tip: start running the script as is with a smoke test i.e. `--n_scenarios 5` to confirm your config and reference set paths are correct before implementing further changes and committing to a full run, which can take foreverr. 





## Step 2 — Satisficing analysis

A policy **satisfices** in a scenario if it meets an acceptable threshold on
**every** objective simultaneously. The **satisficing score** is the fraction
of scenarios in which a policy does this.

**Task 1. Choose your thresholds**

For each of the four objectives, decide what counts as "acceptable" performance.
You must define one threshold per objective and justify your choice. Consider:

- Is there an external standard you can use? For example, the Paris Agreement
  temperature target, which could directly inform your threshold?
- If no external standard exists, you can establish a specific percentile across all policies and scenarios.
- Should some objectives be held to a stricter standard than others? Explain why.

Document your reasoning for each threshold before moving on.


**Task 2. Compute satisficing scores**

A policy *satisfices* in a given scenario if it meets **all four thresholds simultaneously**.

To organize this informtion, you could first populate a boolean array of shape `(n_policies, n_scenarios)` where each entry
is `True` if the policy meets all thresholds in that scenario, and `False` otherwise. Where the columns are the scenarios, and the rows are the policies. So, something like this:
|     | S1    | S2    | S3    | S4    |
|-----|-------|-------|-------|-------|
| P0  | True  | True  | False | True  |
| P1  | False | False | False | False |
| P2  | True  | True  | True  | True  |
| P3  | True  | False | True  | False |
| P4  | False | True  | False | True  |


**Task 3. Compute and report satisficing scores**

For each policy, compute its **satisficing score**: the fraction of scenarios in
which it satisfices all objectives. A score of 1.0 means the policy meets all
thresholds in every scenario; a score of 0.0 means it never does.

Report the following:
- The mean and maximum satisficing score across all policies
- How many policies have a satisficing score of zero (never satisfice)
- The per-objective satisficing rate: for each objective separately, what
  fraction of (policy, scenario) pairs meet that threshold? This tells you
  which objective is the hardest constraint to satisfy.

**Task 4. Visualizing satisfying scores**
Create a heatmap with policies as rows (sorted from highest to lowest
satisficing score) and objectives as columns. Each cell shows the fraction
of scenarios in which that policy meets the threshold for that objective.
**Useful references:**
- [`sns.heatmap()` documentation](https://seaborn.pydata.org/generated/seaborn.heatmap.html) — pay attention to `vmin`, `vmax`, `cmap`, `annot`, and `linewidths`
- [`np.argsort()`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — for sorting row indices by satisficing score

> Tip 1: Use a shared color scale with `vmin=0, vmax=1` so all objectives
are comparable.

> Tip 2: If you decide that dark = meets threshold in most scenarios, then the column with the lighter cells would be the hardest to satisfy.


### Task 1 — Threshold justification [Actor 17: Russian Federation]

The four thresholds below define what counts as "acceptable" performance from Russia's negotiating position. Russia's mandate prioritises minimising abatement costs while maintaining a defensible public position on welfare and climate safety.

---

**1. `welfare` — Total welfare loss ≤ 107.0**

*Basis: 75th percentile of the Pareto reference set.*

The welfare objective ranges from 106.82 to 107.14 across the reference set — an extremely narrow spread of less than 0.3%. Because no external standard exists for total aggregate welfare in JUSTICE units, a percentile-based threshold is the appropriate choice. Setting the threshold at 107.0 (near the 75th percentile) means Russia accepts policies that keep welfare loss within the better-performing three-quarters of the Pareto front. Russia's stated public position is to maximise total welfare, so a lenient threshold is consistent with that framing — Russia does not want to rule out policies purely on welfare grounds when the differences between policies are marginal.

---

**2. `years_above_2C` — Years above 2°C ≤ 100**

*Basis: Paris Agreement 2°C limit (Article 2.1a, UNFCCC 2015), applied pragmatically to the 2015–2300 simulation horizon.*

The Paris Agreement commits signatories to holding warming "well below 2°C above pre-industrial levels." The simulation runs 285 years (2015–2300). A threshold of 0 years above 2°C would be consistent with the letter of Paris but would be unachievable under high-sensitivity FAIR ensemble members regardless of policy. Russia would argue that temporary temperature overshoot followed by drawdown is physically and economically more realistic than permanent sub-2°C trajectories, and that the IPCC itself acknowledges overshoot scenarios in 1.5°C-compatible pathways (IPCC AR6 WG1 Chapter 4). A threshold of 100 years (≈35% of the simulation horizon) is lenient enough to reflect Russia's position that aggressive near-term abatement is not justified, while still excluding the most catastrophic high-warming trajectories. Russia's own damage exposure (large ensemble spread of `damage_fraction` for `rus`) means it is not indifferent to warming — it simply does not accept that the 2°C target must be met in every single climate scenario.

---

**3. `welfare_loss_damage` — Welfare loss from damage ≤ 3,662**

*Basis: 75th percentile of the Pareto reference set.*

No internationally agreed welfare-loss-from-damage threshold exists in the literature in JUSTICE-compatible units. IPCC AR6 WG2 estimates climate damages at up to 7.5% of GDP under business-as-usual and below 4.5% of GDP for well-below-2°C pathways, but these cannot be directly mapped to JUSTICE's welfare units. The reference set ranges from 3,639 to 3,667 — a narrow band. Russia acknowledges its own damage exposure (the actor file explicitly notes uncertainty in `damage_fraction` for `rus`) and would not advocate for a threshold so lenient that catastrophic damage scenarios are accepted. The 75th percentile (3,662) is a moderate threshold: it excludes the worst-performing quarter of the front on damage while not demanding near-optimal damage performance from every scenario.

---

**4. `welfare_loss_abatement` — Welfare loss from abatement ≤ 12,950**

*Basis: 75th percentile of the Pareto reference set; justified by IPCC AR6 WG3 abatement cost benchmarks.*

This is Russia's primary constraint. The reference set shows a strongly right-skewed distribution: most policies (P10–P75) cluster between 12,723 and 12,951, but a long tail reaches 30,677. IPCC AR6 WG3 estimates 2°C-compatible mitigation costs at 1–4% of GDP by 2050 under globally coordinated carbon pricing — a significant but finite economic burden. Russia's fossil fuel export economy means abatement costs fall disproportionately on its industrial base. Setting the threshold at the 75th percentile (12,950) means Russia accepts the lower three-quarters of the abatement cost distribution — a moderate position that is defensible in negotiations without conceding to the most expensive mitigation pathways. Policies in the high-cost tail (above ~13,000 and especially above 20,000) represent economically unjustifiable burdens under Russia's framing.

---

**Summary table**

| Objective | Threshold | Direction | Basis |
|---|---|---|---|
| `welfare` | ≤ 107.0 | Lenient | 75th percentile, public welfare framing |
| `years_above_2C` | ≤ 100 years | Moderate | Paris Agreement 2°C; pragmatic overshoot allowance |
| `welfare_loss_damage` | ≤ 3,662 | Moderate | 75th percentile; Russia has real damage exposure |
| `welfare_loss_abatement` | ≤ 12,950 | Strict | 75th percentile; Russia's core economic interest |

In [ ]:
#YOUR CODE HERE

## Step 3 — Minimax Regret

Satisficing tells you whether a policy is *acceptable*. Minimax regret tells
you which policy has the **smallest worst-case loss** — the one that never
performs catastrophically relative to what was achievable in that scenario.

**Task 1. Compute the per-scenario ideal and anti-ideal.**
For each scenario, find the best and worst value any policy achieves on each
objective. These define the range of performance possible in that scenario.

**Task 2. Compute normalised regret.**
For each (policy, scenario, objective), regret is how far that policy
falls short of the best achievable outcome in that scenario, expressed as a
fraction of the full range. (You can subtract the ideal from each policy's result, then divide by the range (anti-ideal − ideal) computed in Task 1). A regret of 0 means the policy matched the best
possible outcome; a regret of 1 means it achieved the worst possible outcome.

**Task 3. Compute maximum regret per policy.**
For each policy, take the worst-case total regret across all scenarios. This
is the **maximum regret** — the single number that summarises how badly the
policy could perform in the worst climate future.

**Task 4. Identify the minimax-regret policy.**
Sort policies by maximum regret. The policy with the lowest maximum regret is
the most robust choice under this criterion.

**Task 5. Plot a CDF of maximum regret**
Plot the cumulative distribution function of maximum regret across all
policies (x = maximum regret, y = fraction of policies with regret ≤ x).
Mark the minimax-regret policy with a vertical line.

This plot shows the full spread of robustness across the Pareto front —
whether most policies are similarly robust or whether there is a wide range.


In [ ]:
#YOUR CODE HERE

## Reflection Questions

**How does the minimax-regret policy compares to the best satisficing policy? are they the same policy or different ones?**

**What does this tell you about your selected robustness method?**

**Which single policy would you recommend, and why?**
